In [1]:
import random
from rdkit import Chem
from molpher.core import MolpherMol, MolpherAtom
from molpher.core.morphing.operators import MorphingOperator
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions
from rdkit.Chem import rdChemReactions
from rdkit.Chem import rdmolops
from rdkit.Chem import Descriptors  
from molpher.core import ExplorationTree as ETree

class NitrogenSulfation(MorphingOperator):
    def __init__(self):
        super(NitrogenSulfation, self).__init__()
        self._name = "N-Sulfation (Amines 1°, 2°, 3° & Aromatic)"
        self._target_atoms = []
        self.PATTERN = Chem.MolFromSmarts("[N;X3;!$(N-C=O);!$(N-C(=O)N);!$(N-C(=O)O)]")

    def setOriginal(self, mol):
        super(NitrogenSulfation, self).setOriginal(mol)
        self._target_atoms = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._target_atoms.append(match[0])

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None
        
        if not self._target_atoms:
            return MolpherMol(other=rdkit_mol)
        
        nitrogen_idx = random.choice(self._target_atoms)
        
        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            is_tertiary = (rw_mol.GetAtomWithIdx(nitrogen_idx).GetTotalNumHs() == 0)
            
            sulfur_idx = rw_mol.AddAtom(Chem.Atom(16)) # S
            rw_mol.AddBond(nitrogen_idx, sulfur_idx, Chem.BondType.SINGLE)
            
            o1_idx = rw_mol.AddAtom(Chem.Atom(8)) # =O
            rw_mol.AddBond(sulfur_idx, o1_idx, Chem.BondType.DOUBLE)
            
            o2_idx = rw_mol.AddAtom(Chem.Atom(8)) # =O
            rw_mol.AddBond(sulfur_idx, o2_idx, Chem.BondType.DOUBLE)
            
            o3_idx = rw_mol.AddAtom(Chem.Atom(8)) # -OH
            rw_mol.AddBond(sulfur_idx, o3_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
            
            for idx in [o1_idx, o2_idx, o3_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                atom.SetFormalCharge(0)
            
            n_atom = new_mol.GetAtomWithIdx(nitrogen_idx)
            n_atom.SetNoImplicit(False)
            n_atom.SetNumExplicitHs(0)
            
            if is_tertiary:
                
                n_atom.SetFormalCharge(1)
            else:
                
                n_atom.SetFormalCharge(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name
    
nitrogen_sulfation = NitrogenSulfation()

valid_tests = {
        "1° Αμίνη (Μεθυλαμίνη)": "CN",
        "2° Αμίνη (Διμεθυλαμίνη)": "CNC",
        "3° Αμίνη (Τριαιθυλαμίνη)": "CCN(CC)CC",
        "Αρωματική Αμίνη (Ανιλίνη)": "Nc1ccccc1"
}
    
trap_tests = {
        "Αμίδιο (Παρακεταμόλη)": "CC(=O)Nc1ccc(O)cc1",
        "Ουρία Trap": "CNC(=O)NC",
        "Καρβαμιδικό Trap": "CNC(=O)OC",
        "Πυριδίνη (Αρωματικό άζωτο εντός δακτυλίου)": "c1ccncc1"
    }
    
print("=== TESTING VALID AMINES (EXPECTED SUCCESS) ===")
for name, smiles in valid_tests.items():
    mol = MolpherMol(smiles)
    nitrogen_sulfation.setOriginal(mol)
    product = nitrogen_sulfation.morph()
        
    success = (product.getSMILES() != mol.getSMILES())

    print(f"{name}:")
    print(f"  SOURCE: {mol.getSMILES()}")
    print(f"  PRODUCT: {product.getSMILES() if success else 'No change (Failed)'}")
    print("-" * 50)
        
print("\n=== TESTING TRAPS & PROTECTIONS (EXPECTED SAFE) ===")
for name, smiles in trap_tests.items():
    mol = MolpherMol(smiles)
    nitrogen_sulfation.setOriginal(mol)
    product = nitrogen_sulfation.morph()
    safe = (product.getSMILES() == mol.getSMILES())

    print(f"{name}:")
    print(f"  SOURCE: {mol.getSMILES()}")
    print(f"  STATUS: {'SAFE (Passed)' if safe else 'VULNERABLE (Failed)'}")
    print("-" * 50)

=== TESTING VALID AMINES (EXPECTED SUCCESS) ===
1° Αμίνη (Μεθυλαμίνη):
  SOURCE: CN
  PRODUCT: CNS(=O)(=O)O
--------------------------------------------------
2° Αμίνη (Διμεθυλαμίνη):
  SOURCE: CNC
  PRODUCT: CN(C)S(=O)(=O)O
--------------------------------------------------
3° Αμίνη (Τριαιθυλαμίνη):
  SOURCE: CCN(CC)CC
  PRODUCT: CC[N+](CC)(CC)S(=O)(=O)O
--------------------------------------------------
Αρωματική Αμίνη (Ανιλίνη):
  SOURCE: NC1=CC=CC=C1
  PRODUCT: O=S(=O)(O)NC1=CC=CC=C1
--------------------------------------------------

=== TESTING TRAPS & PROTECTIONS (EXPECTED SAFE) ===
Αμίδιο (Παρακεταμόλη):
  SOURCE: CC(=O)NC1=CC=C(O)C=C1
  STATUS: SAFE (Passed)
--------------------------------------------------
Ουρία Trap:
  SOURCE: CNC(=O)NC
  STATUS: SAFE (Passed)
--------------------------------------------------
Καρβαμιδικό Trap:
  SOURCE: CNC(=O)OC
  STATUS: SAFE (Passed)
--------------------------------------------------
Πυριδίνη (Αρωματικό άζωτο εντός δακτυλίου):
  SOURCE: